---
# 📂 Notebook: `limpieza_olist.ipynb`
---

In [ ]:
import os
import warnings
import unicodedata

import numpy as np
import pandas as pd

# Silenciar solo ruido conocido: cambios de API en pandas y categorias deprecadas.
# Otros warnings (p.ej. SettingWithCopy real) seguiran visibles.
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

# Rutas relativas al notebook (../CSV en la raiz del proyecto).
# Para re-ejecutar la limpieza desde cero, los CSV crudos de Olist deben estar en DATA_PATH.
DATA_PATH   = os.environ.get('DATA_PATH', '../CSV/')
OUTPUT_PATH = os.environ.get('OUTPUT_PATH', '../CSV/')
os.makedirs(OUTPUT_PATH, exist_ok=True)


---
## 1. `olist_customers_dataset`

In [ ]:
customers = pd.read_csv(DATA_PATH + 'olist_customers_dataset.csv',dtype={'customer_zip_code_prefix': str})

print(f'Filas : {customers.shape[0]:,}  |  Columnas : {customers.shape[1]}')
print(f'\nTipos de datos:\n{customers.dtypes}')
print(f'\nNulos por columna:\n{customers.isnull().sum()}')
print(f'\nDuplicados (fila completa): {customers.duplicated().sum():,}')
print(f'Duplicados en customer_id:  {customers["customer_id"].duplicated().sum():,}')
customers.head()

Filas: 99,441  |  Columnas: 5

Tipos de datos:
customer_id                 str
customer_unique_id          str
customer_zip_code_prefix    str
customer_city               str
customer_state              str
dtype: object

Nulos por columna:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

Duplicados (fila completa): 0
Duplicados en customer_id:  0


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [ ]:
customers_clean = customers.copy()

customers_clean['customer_zip_code_prefix'] = (customers_clean['customer_zip_code_prefix'].str.strip().str.zfill(5))

customers_clean['customer_city'] = (customers_clean['customer_city'].str.strip().str.title())

print(customers_clean[['customer_zip_code_prefix', 'customer_city']].head(5).to_string())
print(f'\nFilas antes:  {customers.shape[0]:,}')
print(f'Filas después: {customers_clean.shape[0]:,}')
if customers.shape[0] == customers_clean.shape[0]:
    print('No se eliminaron filas')
else:
    print(f'Se eliminaron {customers.shape[0]-customers_clean.shape[0]:,} filas')

  customer_zip_code_prefix          customer_city
0                    14409                 Franca
1                    09790  Sao Bernardo Do Campo
2                    01151              Sao Paulo
3                    08775        Mogi Das Cruzes
4                    13056               Campinas

Filas antes:  99,441
Filas después: 99,441
No se eliminaron filas


In [ ]:
customers_clean.to_csv(OUTPUT_PATH + 'olist_customers_clean.csv', index=False)
customers_clean.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,Franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,Sao Bernardo Do Campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,Sao Paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,Mogi Das Cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,Campinas,SP


---
## 2. `olist_order_items_dataset`

In [ ]:
order_items = pd.read_csv(DATA_PATH + 'olist_order_items_dataset.csv')

print(f'Filas: {order_items.shape[0]:,}  |  Columnas: {order_items.shape[1]}')
print(f'\nTipos de datos:\n{order_items.dtypes}')
print(f'\nNulos por columna:\n{order_items.isnull().sum()}')
print(f'\nDuplicados (fila completa): {order_items.duplicated().sum():,}')
print(f'\nEstadísticas de precio y flete:')
print(order_items[['price', 'freight_value']].describe())
print(f'\nRegistros con flete = 0: {(order_items["freight_value"] == 0).sum():,}')
print(f'Registros con precio = 0: {(order_items["price"] == 0).sum():,}')
order_items.head()

Filas: 112,650  |  Columnas: 7

Tipos de datos:
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object

Nulos por columna:
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

Duplicados (fila completa): 0

Estadísticas de precio y flete:
               price  freight_value
count  112650.000000  112650.000000
mean      120.653739      19.990320
std       183.633928      15.806405
min         0.850000       0.000000
25%        39.900000      13.080000
50%        74.990000      16.260000
75%       134.900000      21.150000
max      6735.000000     409.680000

Registros con flete = 0: 383
Registros con precio = 0: 0


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [ ]:
order_items_clean = order_items.copy()

order_items_clean['shipping_limit_date'] = pd.to_datetime(order_items_clean['shipping_limit_date'],errors='coerce')

nat_count = order_items_clean['shipping_limit_date'].isna().sum()
print(f'Fechas que no pudieron convertirse (NaT): {nat_count}')

Fechas que no pudieron convertirse (NaT): 0


In [ ]:
order_items_clean.to_csv(OUTPUT_PATH + 'olist_order_items_clean.csv', index=False)
order_items_clean.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


---
## 3. `olist_geolocation_dataset`

In [ ]:
geo = pd.read_csv(DATA_PATH + 'olist_geolocation_dataset.csv',dtype={'geolocation_zip_code_prefix': str})

print(f'Filas: {geo.shape[0]:,}  |  Columnas: {geo.shape[1]}')
print(f'\nTipos de datos:\n{geo.dtypes}')
print(f'\nNulos por columna:\n{geo.isnull().sum()}')
print(f'\nDuplicados (fila completa): {geo.duplicated().sum():,}')
print(f'Zips únicos: {geo["geolocation_zip_code_prefix"].nunique():,}')
print(f'\nEstadísticas de coordenadas:')
print(geo[['geolocation_lat', 'geolocation_lng']].describe())
geo.head()

Filas: 1,000,163  |  Columnas: 5

Tipos de datos:
geolocation_zip_code_prefix        str
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str
dtype: object

Nulos por columna:
geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

Duplicados (fila completa): 261,831
Zips únicos: 19,015

Estadísticas de coordenadas:
       geolocation_lat  geolocation_lng
count     1.000163e+06     1.000163e+06
mean     -2.117615e+01    -4.639054e+01
std       5.715866e+00     4.269748e+00
min      -3.660537e+01    -1.014668e+02
25%      -2.360355e+01    -4.857317e+01
50%      -2.291938e+01    -4.663788e+01
75%      -1.997962e+01    -4.376771e+01
max       4.506593e+01     1.211054e+02


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,01037,-23.545621,-46.639292,sao paulo,SP
1,01046,-23.546081,-46.644820,sao paulo,SP
2,01046,-23.546129,-46.642951,sao paulo,SP
3,01041,-23.544392,-46.639499,sao paulo,SP
4,01035,-23.541578,-46.641607,sao paulo,SP


In [ ]:
geo_clean = geo.drop_duplicates()
print(f'Filas antes:  {geo.shape[0]:,}')
print(f'Filas después: {geo_clean.shape[0]:,}')
print(f'Eliminadas:   {geo.shape[0] - geo_clean.shape[0]:,}')

Filas antes:  1,000,163
Filas después: 738,332
Eliminadas:   261,831


In [ ]:
# Límites geográficos de Brasil (chance estén mal)
LAT_MIN, LAT_MAX = -33.75, 5.27
LNG_MIN, LNG_MAX = -73.98, -34.79

fuera_rango = geo_clean[(geo_clean['geolocation_lat'] < LAT_MIN)|(geo_clean['geolocation_lat'] > LAT_MAX)|(geo_clean['geolocation_lng'] < LNG_MIN)|(geo_clean['geolocation_lng'] > LNG_MAX)]
print(f'Registros con coordenadas fuera de Brasil: {len(fuera_rango):,}')
print(fuera_rango[['geolocation_zip_code_prefix','geolocation_lat','geolocation_lng','geolocation_city']].head(10))

geo_clean = geo_clean[
    (geo_clean['geolocation_lat'] >= LAT_MIN) &
    (geo_clean['geolocation_lat'] <= LAT_MAX) &
    (geo_clean['geolocation_lng'] >= LNG_MIN) &
    (geo_clean['geolocation_lng'] <= LNG_MAX)
]
print(f'\nFilas después de filtrar coordenadas: {geo_clean.shape[0]:,}')

Registros con coordenadas fuera de Brasil: 33
       geolocation_zip_code_prefix  geolocation_lat  geolocation_lng  \
387565                       18243        28.008978       -15.536867   
513631                       28165        41.614052        -8.411675   
513643                       28155       -34.586422       -58.732101   
513754                       28155        42.439286        13.820214   
514429                       28333        38.381672        -6.328200   
516682                       28595        43.684961        -7.411080   
538512                       29654        29.409252       -98.484121   
538557                       29654        21.657547      -101.466766   
585242                       35179        25.995203       -98.078544   
585260                       35179        25.995245       -98.078533   

               geolocation_city  
387565  bom retiro da esperanca  
513631      vila nova de campos  
513643              santa maria  
513754              santa

In [ ]:
def normalizar_texto(texto):
    if pd.isna(texto):
        return texto
    texto_normalizado = unicodedata.normalize('NFKD', str(texto))
    texto_sin_acentos = ''.join(
        c for c in texto_normalizado
        if not unicodedata.combining(c)
    )
    return texto_sin_acentos.lower().strip()

# Muestra antes
print('Muestra ANTES:')
print(geo_clean['geolocation_city'].unique()[:8])

# Aplicar normalización
geo_clean = geo_clean.copy()
geo_clean['geolocation_city'] = geo_clean['geolocation_city'].apply(normalizar_texto)

# Muestra después
print('\nMuestra DESPUÉS:')
print(geo_clean['geolocation_city'].unique()[:8])

# Verificar que 'são paulo' y 'sao paulo' ahora son iguales
print(f'\nCiudades únicas antes de normalizar: (ya aplicado en paso anterior)')
print(f'Ciudades únicas después de normalizar: {geo_clean["geolocation_city"].nunique():,}')

Muestra ANTES:
<StringArray>
[            'sao paulo',             'são paulo', 'sao bernardo do campo',
               'jundiaí',       'taboão da serra',              'sãopaulo',
                    'sp',            'sa£o paulo']
Length: 8, dtype: str

Muestra DESPUÉS:
<StringArray>
[            'sao paulo', 'sao bernardo do campo',               'jundiai',
       'taboao da serra',              'saopaulo',                    'sp',
            'sa£o paulo',   'sao jose dos campos']
Length: 8, dtype: str

Ciudades únicas antes de normalizar: (ya aplicado en paso anterior)
Ciudades únicas después de normalizar: 5,963


In [ ]:
geo_clean['geolocation_zip_code_prefix'] = (geo_clean['geolocation_zip_code_prefix'].str.strip().str.zfill(5))

geo_agg = geo_clean.groupby('geolocation_zip_code_prefix').agg(
    geolocation_lat   = ('geolocation_lat',  'mean'),
    geolocation_lng   = ('geolocation_lng',  'mean'),
    geolocation_city  = ('geolocation_city',  lambda x: x.mode()[0]),
    geolocation_state = ('geolocation_state', lambda x: x.mode()[0])
).reset_index()

geo_agg['geolocation_city'] = geo_agg['geolocation_city'].str.title()

print(f'Zips únicos en el resultado: {geo_agg.shape[0]:,}')
print(f'\nResumen del proceso:')
print(f'  Filas originales:          {geo.shape[0]:,}')
print(f'  Tras eliminar duplicados:  {geo.shape[0] - 261831:,}')
print(f'  Tras filtrar coordenadas:  {geo_clean.shape[0]:,}')
print(f'  Tras agregar por zip:      {geo_agg.shape[0]:,}  ← tabla final')
geo_agg.head()

Zips únicos en el resultado: 19,010

Resumen del proceso:
  Filas originales:          1,000,163
  Tras eliminar duplicados:  738,332
  Tras filtrar coordenadas:  738,299
  Tras agregar por zip:      19,010  ← tabla final


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,01001,-23.550227,-46.634039,Sao Paulo,SP
1,01002,-23.547657,-46.634991,Sao Paulo,SP
2,01003,-23.549000,-46.635582,Sao Paulo,SP
3,01004,-23.549829,-46.634792,Sao Paulo,SP
4,01005,-23.549547,-46.636406,Sao Paulo,SP


In [ ]:
geo_agg.to_csv(OUTPUT_PATH + 'olist_geolocation_clean.csv', index=False)
geo_agg.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,01001,-23.550227,-46.634039,Sao Paulo,SP
1,01002,-23.547657,-46.634991,Sao Paulo,SP
2,01003,-23.549000,-46.635582,Sao Paulo,SP
3,01004,-23.549829,-46.634792,Sao Paulo,SP
4,01005,-23.549547,-46.636406,Sao Paulo,SP


---
# 📂 Notebook: `limpieza_orders_products.ipynb`
---

# Documentación del Proceso de Limpieza: Products y Orders

## 1. Limpieza del Dataset de Productos (`olist_products_dataset.csv`)
El objetivo principal en esta tabla fue preservar todos los registros (IDs de productos) para no perder transacciones en el modelo OLAP final. Se optó por la imputación de valores nulos en lugar de la eliminación de filas.

* **Normalización de Categorías (`product_category_name`):**
  * Se identificaron 610 valores nulos y se rellenaron con la etiqueta genérica `'Sin Categoria'`.
  * Se aplicó limpieza de formato de texto: se reemplazaron los guiones bajos (`_`) por espacios y se aplicó formato de título (Title Case) para estandarizar la visualización en los Dashboards (ej. pasó de `esporte_lazer` a `Esporte Lazer`).
* **Imputación de Variables Numéricas:**
  * Las columnas correspondientes a dimensiones, peso y cantidad de fotografías contenían valores nulos esporádicos.
  * Estos vacíos se imputaron utilizando la **mediana** de cada respectiva columna. Se eligió la mediana por encima de la media geométrica o el promedio para evitar que la asimetría de valores atípicos (outliers) distorsionara los datos.

## 2. Limpieza del Dataset de Pedidos (`olist_orders_dataset.csv`)
El enfoque para la tabla de pedidos transaccionales fue estandarizar los datos categóricos y asegurar la integridad de los tipos de datos temporales (Time Series) para el posterior análisis logístico.

* **Estandarización de Estatus (`order_status`):**
  * Se eliminaron espacios en blanco accidentales (`strip`) y se capitalizó el texto para mantener consistencia en agrupaciones futuras (ej. `Delivered`, `Canceled`).
* **Conversión de Fechas (Formato Datetime):**
  * Las 5 columnas relacionadas con marcas de tiempo (compra, aprobación, envío a paquetería, entrega y fecha estimada) se convirtieron rigurosamente de texto (String) a formato `datetime64`.
  * **Decisión de Negocio sobre Nulos:** Se conservaron intencionalmente los valores nulos generados en las columnas de entrega (`order_delivered_customer_date`, etc.). Esto respeta la naturaleza operativa del negocio, ya que un pedido con estatus `Canceled` o `Invoiced` lógicamente carece de una marca de tiempo de entrega final.
* **Ingeniería de Características (Feature Engineering):**
  * Se extrajo y generó una nueva columna llamada **`fecha_compra_dia`** a partir del timestamp original, aislando únicamente la fecha (YYYY-MM-DD).
  * *Propósito:* Esta característica fue construida estratégicamente para servir como llave foránea (JOIN key) al momento de integrar **fuentes exógenas**, permitiendo cruzar cada pedido con el clima exacto de ese día.

In [ ]:
import pandas as pd
import numpy as np
import unicodedata
import os
import requests
from sqlalchemy import create_engine

warnings.filterwarnings('ignore')


# ==========================================
# 0. CONFIGURACIÓN INICIAL
# ==========================================

In [ ]:
# './' le dice a Python: "busca en esta misma carpeta"
DATA_PATH = './'

print("Cargando archivos...")

# Leemos cada archivo CSV
products    = pd.read_csv(DATA_PATH + 'olist_products_dataset.csv')
orders      = pd.read_csv(DATA_PATH + 'olist_orders_dataset.csv')

print("¡Todos los archivos se cargaron con éxito!")

Cargando archivos...
¡Todos los archivos se cargaron con éxito!


# ==========================================
# 1. LIMPIEZA DE PRODUCTS
# ==========================================

In [ ]:
print("Limpiando Products...")
products_clean = products.copy()

Limpiando Products...


In [ ]:
# Rellenar categorías vacías y darle formato (sin guiones bajos y con Mayúsculas)
products_clean['product_category_name'] = products_clean['product_category_name'].fillna('sin_categoria')
products_clean['product_category_name'] = products_clean['product_category_name'].str.replace('_', ' ').str.title()

In [ ]:
# Rellenar dimensiones/pesos vacíos con la mediana para no perder el registro
cols_numericas = ['product_name_lenght', 'product_description_lenght', 'product_photos_qty',
                  'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
for col in cols_numericas:
    products_clean[col] = products_clean[col].fillna(products_clean[col].median())

print(f"✅ Products limpio (Filas: {products_clean.shape[0]:,})")

✅ Products limpio (Filas: 32,951)


# ==========================================
# 2. LIMPIEZA DE ORDERS
# ==========================================

In [ ]:
print("\nLimpiando Orders...")
orders_clean = orders.copy()


Limpiando Orders...


In [ ]:
# Normalizar el texto del estatus
orders_clean['order_status'] = orders_clean['order_status'].str.strip().str.title()

In [ ]:
# Convertir textos a formato Fecha (Datetime)
date_columns = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
                'order_delivered_customer_date', 'order_estimated_delivery_date']

for col in date_columns:
    orders_clean[col] = pd.to_datetime(orders_clean[col], errors='coerce')

In [ ]:
# (Opcional pero recomendado) Extraemos solo el día de la compra en una columna nueva
# Esto te servirá muchísimo cuando decidas conectar el clima más adelante
orders_clean['fecha_compra_dia'] = orders_clean['order_purchase_timestamp'].dt.date

print(f"✅ Orders limpio (Filas: {orders_clean.shape[0]:,})")

✅ Orders limpio (Filas: 99,441)


# ==========================================
# 3. EXPORTAR ARCHIVOS LIMPIOS
# ==========================================

In [ ]:
import os

print("\nExportando archivos...")

# Definimos la ruta aquí mismo para evitar el NameError
OUTPUT_PATH = './data_clean/'
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Guardamos los archivos
products_clean.to_csv(OUTPUT_PATH + 'olist_products_clean.csv', index=False)
orders_clean.to_csv(OUTPUT_PATH + 'olist_orders_clean.csv', index=False)

print(f"¡Listo! Archivos guardados en la carpeta: {OUTPUT_PATH}")


Exportando archivos...
¡Listo! Archivos guardados en la carpeta: ./data_clean/


---
# 📂 Notebook: `limpieza_payments.ipynb`
---

# Limpieza de olist_order_payments_dataset.csv

Este notebook carga, limpia y valida el dataset de pagos de Olist.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [ ]:
# Carga del dataset
file_path = 'olist_order_payments_dataset.csv'
df = pd.read_csv(file_path)

print(f'Filas iniciales: {len(df):,}')
display(df.head())
display(df.info())

Filas iniciales: 103,886


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


<class 'pandas.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  str    
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  str    
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), str(2)
memory usage: 4.0 MB


None

In [ ]:
# Diagnostico inicial
missing = df.isna().sum().to_frame('missing_count')
missing['missing_pct'] = (missing['missing_count'] / len(df) * 100).round(2)

print('Duplicados exactos:', int(df.duplicated().sum()))
display(missing)
display(df.describe(include='all').T)

Duplicados exactos: 0


,missing_count,missing_pct
order_id,0,0.0
payment_sequential,0,0.0
payment_type,0,0.0
payment_installments,0,0.0
payment_value,0,0.0


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
order_id,103886,99440,fa65dad1b0e818e3ccc5cb0e39231352,29,NaN,NaN,NaN,NaN,NaN,NaN,NaN
payment_sequential,103886.0,NaN,NaN,NaN,1.092679,0.706584,1.0,1.0,1.0,1.0,29.0
payment_type,103886,5,credit_card,76795,NaN,NaN,NaN,NaN,NaN,NaN,NaN
payment_installments,103886.0,NaN,NaN,NaN,2.853349,2.687051,0.0,1.0,1.0,4.0,24.0
payment_value,103886.0,NaN,NaN,NaN,154.10038,217.494064,0.0,56.79,100.0,171.8375,13664.08


In [ ]:
# Limpieza
df_clean = df.copy()

# 1) Estandarizar nombres de columnas
df_clean.columns = [c.strip().lower() for c in df_clean.columns]

# 2) Limpiar campos de texto
df_clean['order_id'] = df_clean['order_id'].astype('string').str.strip().str.lower()
df_clean['payment_type'] = df_clean['payment_type'].astype('string').str.strip().str.lower()

# 3) Convertir tipos numericos
num_cols = ['payment_sequential', 'payment_installments', 'payment_value']
for col in num_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# 4) Eliminar duplicados exactos
before_dup = len(df_clean)
df_clean = df_clean.drop_duplicates()
removed_dup = before_dup - len(df_clean)

# 5) Mantener la mayor cantidad de datos: solo eliminar filas sin order_id usable
missing_order_id = df_clean['order_id'].isna() | df_clean['order_id'].isin(['', 'nan'])
removed_missing_order_id = int(missing_order_id.sum())
df_clean = df_clean[~missing_order_id].copy()

# 6) Imputacion conservadora para no perder registros
na_seq = int(df_clean['payment_sequential'].isna().sum())
na_inst = int(df_clean['payment_installments'].isna().sum())
na_val = int(df_clean['payment_value'].isna().sum())
df_clean['payment_sequential'] = df_clean['payment_sequential'].fillna(1)
df_clean['payment_installments'] = df_clean['payment_installments'].fillna(1)
df_clean['payment_value'] = df_clean['payment_value'].fillna(0)

# 7) Eliminar solo valores imposibles (negativos)
before_range = len(df_clean)
df_clean = df_clean[(df_clean['payment_sequential'] >= 1) &
                    (df_clean['payment_installments'] >= 0) &
                    (df_clean['payment_value'] >= 0)].copy()
removed_invalid_values = before_range - len(df_clean)

# 8) Tipos finales
df_clean['payment_sequential'] = df_clean['payment_sequential'].astype('int64')
df_clean['payment_installments'] = df_clean['payment_installments'].astype('int64')
df_clean['payment_value'] = df_clean['payment_value'].astype('float64')
df_clean['payment_type'] = df_clean['payment_type'].replace({'': np.nan, 'nan': np.nan}).fillna('unknown')

print(f'Duplicados eliminados: {removed_dup}')
print(f'Filas sin order_id eliminadas: {removed_missing_order_id}')
print(f'Nulos imputados en payment_sequential: {na_seq}')
print(f'Nulos imputados en payment_installments: {na_inst}')
print(f'Nulos imputados en payment_value: {na_val}')
print(f'Filas eliminadas por valores imposibles: {removed_invalid_values}')
print(f'Filas finales: {len(df_clean):,}')

Duplicados eliminados: 0
Filas sin order_id eliminadas: 0
Nulos imputados en payment_sequential: 0
Nulos imputados en payment_installments: 0
Nulos imputados en payment_value: 0
Filas eliminadas por valores imposibles: 0
Filas finales: 103,886


In [ ]:
# Revision final de calidad
display(df_clean.head())
display(df_clean.isna().sum().to_frame('missing_count'))
display(df_clean.describe(include='all').T)

print('Duplicados exactos finales:', int(df_clean.duplicated().sum()))

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


,missing_count
order_id,0
payment_sequential,0
payment_type,0
payment_installments,0
payment_value,0


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
order_id,103886,99440,fa65dad1b0e818e3ccc5cb0e39231352,29,NaN,NaN,NaN,NaN,NaN,NaN,NaN
payment_sequential,103886.0,NaN,NaN,NaN,1.092679,0.706584,1.0,1.0,1.0,1.0,29.0
payment_type,103886,5,credit_card,76795,NaN,NaN,NaN,NaN,NaN,NaN,NaN
payment_installments,103886.0,NaN,NaN,NaN,2.853349,2.687051,0.0,1.0,1.0,4.0,24.0
payment_value,103886.0,NaN,NaN,NaN,154.10038,217.494064,0.0,56.79,100.0,171.8375,13664.08


Duplicados exactos finales: 0


In [ ]:
# Exportar dataset limpio
output_path = 'olist_order_payments_dataset_clean.csv'
df_clean.to_csv(output_path, index=False)
print(f'Archivo guardado en: {output_path}')

Archivo guardado en: olist_order_payments_dataset_clean.csv


In [ ]:
# Carga del dataset
file_path = 'olist_order_payments_dataset_clean.csv'
df = pd.read_csv(file_path)

print(f'Filas iniciales: {len(df):,}')
display(df.head())
display(df.info())

Filas iniciales: 103,886


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


<class 'pandas.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  str    
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  str    
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), str(2)
memory usage: 4.0 MB


None

---
# 📂 Notebook: `limpieza_product_category.ipynb`
---

# Limpieza de `product_category_name_translation.csv`

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
# Carga de datos
# dtype str en seller_zip_code_prefix para preservar ceros iniciales
# Se prueban varias rutas para compatibilidad local, Docker y distintos CWD de Jupyter
import os
_FILENAME = 'product_category_name_translation.csv'
_CANDIDATES = [
    os.path.join('..', 'data', _FILENAME),   # CWD = notebooks/
    os.path.join('data', _FILENAME),          # CWD = raíz del proyecto
    os.path.join('work', 'data', _FILENAME),  # CWD = /home/jovyan (Docker)
]

_data_path = next((p for p in _CANDIDATES if os.path.exists(p)), None)
if _data_path is None:
    raise FileNotFoundError(
        f"No se encontró '{_FILENAME}' en ninguna ruta candidata: {_CANDIDATES}"
    )

print(f"Cargando datos desde: {os.path.abspath(_data_path)}")
df = pd.read_csv(_data_path, encoding='utf-8-sig')
df.head()

Cargando datos desde: /home/jovyan/work/data/product_category_name_translation.csv


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


## 1. Exploración inicial

In [ ]:
print("Dimensiones:", df.shape)
print("\nTipos de datos:")
print(df.dtypes)
print("\nColumnas:", df.columns.tolist())
print("\n=== Valores nulos por columna ===")
print(df.isnull().sum())
print("\nFilas duplicadas:", df.duplicated().sum())
print("Duplicados en product_category_name:", df['product_category_name'].duplicated().sum())
print("Duplicados en product_category_name_english:", df['product_category_name_english'].duplicated().sum())

Dimensiones: (71, 2)

Tipos de datos:
product_category_name            object
product_category_name_english    object
dtype: object

Columnas: ['product_category_name', 'product_category_name_english']

=== Valores nulos por columna ===
product_category_name            0
product_category_name_english    0
dtype: int64

Filas duplicadas: 0
Duplicados en product_category_name: 0
Duplicados en product_category_name_english: 0


In [ ]:
df

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor
5,esporte_lazer,sports_leisure
6,perfumaria,perfumery
7,utilidades_domesticas,housewares
8,telefonia,telephony
9,relogios_presentes,watches_gifts


## 2. Detección de erratas en `product_category_name_english`

La columna en portugués está limpia (snake_case, sin caracteres especiales). Sin embargo, la columna en inglés contiene erratas ortográficas que se detectan a continuación.

In [ ]:
ERRATAS = {
    'costruction':  'construction',
    'fashio_':      'fashion_',
    'confort':      'comfort',
    'craftmanship': 'craftsmanship',
}

print("Erratas detectadas en product_category_name_english:\n")
en = df['product_category_name_english']
for typo, fix in ERRATAS.items():
    matches = en[en.str.contains(typo, na=False)]
    if not matches.empty:
        for val in matches:
            corrected = val.replace(typo, fix)
            print(f"  {val!r:40s}  ->  {corrected!r}")
print(f"\nTotal de valores afectados: {sum(en.str.contains(t, na=False).sum() for t in ERRATAS)}")

Erratas detectadas en product_category_name_english:

  'costruction_tools_garden'                ->  'construction_tools_garden'
  'costruction_tools_tools'                 ->  'construction_tools_tools'
  'fashio_female_clothing'                  ->  'fashion_female_clothing'
  'home_confort'                            ->  'home_comfort'
  'arts_and_craftmanship'                   ->  'arts_and_craftsmanship'

Total de valores afectados: 5


## 3. Corrección de erratas

In [ ]:
df['english_original'] = df['product_category_name_english']

for typo, fix in ERRATAS.items():
    df['product_category_name_english'] = (
        df['product_category_name_english'].str.replace(typo, fix, regex=False)
    )

# Mostrar cambios realizados
mascara = df['product_category_name_english'] != df['english_original']
cambios = df[mascara][['english_original', 'product_category_name_english']].copy()
cambios.columns = ['original', 'corregido']
print(f"Erratas corregidas: {len(cambios)}")
cambios.reset_index(drop=True)

Erratas corregidas: 5


,original,corregido
0,costruction_tools_garden,construction_tools_garden
1,home_confort,home_comfort
2,costruction_tools_tools,construction_tools_tools
3,fashio_female_clothing,fashion_female_clothing
4,arts_and_craftmanship,arts_and_craftsmanship


## 4. Reporte final

In [ ]:
df_clean = df.drop(columns=['english_original'])

print("=" * 50)
print("REPORTE DE LIMPIEZA")
print("=" * 50)
print(f"\nFilas totales       : {len(df_clean)}")
print(f"Columnas            : {df_clean.columns.tolist()}")
print(f"\nValores nulos:")
print('Nulos por columna en df_clean:')
print(df_clean.isnull().sum().to_string())
print(f"\nDuplicados          : {df_clean.duplicated().sum()}")
print(f"Erratas corregidas  : {mascara.sum()}")

# Verificar que ya no hay erratas
residual = 0
for typo in ERRATAS:
    residual += df_clean['product_category_name_english'].str.contains(typo, na=False).sum()
print(f"Erratas residuales  : {residual}")

REPORTE DE LIMPIEZA

Filas totales       : 71
Columnas            : ['product_category_name', 'product_category_name_english']

Valores nulos:
product_category_name            0
product_category_name_english    0

Duplicados          : 0
Erratas corregidas  : 5
Erratas residuales  : 0


## 5. Exportar dataset limpio

In [ ]:
# Guardar sin BOM (encoding utf-8 estándar)
output_path = os.path.join(os.path.dirname(_data_path), 'product_category_name_translation_clean.csv')
df_clean.to_csv(output_path, index=False, encoding='utf-8')
print(f"Dataset limpio guardado en: {os.path.abspath(output_path)}")
print(f"Forma final: {df_clean.shape}")

Dataset limpio guardado en: /home/jovyan/work/data/product_category_name_translation_clean.csv
Forma final: (71, 2)


---
# 📂 Notebook: `limpieza_reviews.ipynb`
---

# Limpieza de olist_order_reviews_dataset.csv

Flujo conservador para limpiar reviews sin eliminar datos validos de mas.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)

In [ ]:
# Carga del dataset
file_path = 'olist_order_reviews_dataset.csv'
df = pd.read_csv(file_path)

print(f'Filas iniciales: {len(df):,}')
display(df.head())
display(df.info())

Filas iniciales: 99,224


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa,2018-03-01 00:00:00,2018-03-02 10:26:53


<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   review_id                99224 non-null  str  
 1   order_id                 99224 non-null  str  
 2   review_score             99224 non-null  int64
 3   review_comment_title     11568 non-null  str  
 4   review_comment_message   40977 non-null  str  
 5   review_creation_date     99224 non-null  str  
 6   review_answer_timestamp  99224 non-null  str  
dtypes: int64(1), str(6)
memory usage: 5.3 MB


None

In [ ]:
# Diagnostico de precision (para decidir limpieza sin perder datos validos)
n = len(df)
df_diag = df.copy()

# 1) Nulos por columna con clasificacion de criticidad
missing = df_diag.isna().sum().to_frame('missing_count')
missing['missing_pct'] = (missing['missing_count'] / n * 100).round(2)
missing['criticidad'] = 'alta'
missing.loc[['review_comment_title', 'review_comment_message'], 'criticidad'] = 'baja (campo opcional)'

# 2) Normalizacion minima de llaves solo para diagnosticar
for col in ['review_id', 'order_id']:
    df_diag[col] = df_diag[col].astype('string').str.strip().str.lower()

missing_review_id = int((df_diag['review_id'].isna() | df_diag['review_id'].isin(['', 'nan'])).sum())
missing_order_id = int((df_diag['order_id'].isna() | df_diag['order_id'].isin(['', 'nan'])).sum())
dup_exact = int(df_diag.duplicated().sum())
dup_review_id = int(df_diag.duplicated(subset=['review_id']).sum())
dup_pair = int(df_diag.duplicated(subset=['review_id', 'order_id']).sum())

# 3) Consistencia de review_score
score_num = pd.to_numeric(df_diag['review_score'], errors='coerce')
invalid_score_type = int(score_num.isna().sum())
score_out_of_range = int(((score_num < 1) | (score_num > 5)).fillna(False).sum())
score_non_integer = int(((score_num % 1 != 0) & score_num.notna()).sum())

# 4) Calidad temporal
creation_dt = pd.to_datetime(df_diag['review_creation_date'], errors='coerce')
answer_dt = pd.to_datetime(df_diag['review_answer_timestamp'], errors='coerce')
invalid_creation_date = int(creation_dt.isna().sum())
invalid_answer_date = int(answer_dt.isna().sum())
answer_before_creation = int(((answer_dt < creation_dt) & answer_dt.notna() & creation_dt.notna()).sum())

# 5) Texto libre: faltantes esperados y placeholders
for col in ['review_comment_title', 'review_comment_message']:
    df_diag[col] = df_diag[col].astype('string').str.replace(r'\\s+', ' ', regex=True).str.strip()
placeholder_pattern = r'^(na|n/a|null|none|-)$'
placeholder_title = int(df_diag['review_comment_title'].str.fullmatch(placeholder_pattern, case=False, na=False).sum())
placeholder_message = int(df_diag['review_comment_message'].str.fullmatch(placeholder_pattern, case=False, na=False).sum())
missing_title = int(df_diag['review_comment_title'].isna().sum())
missing_message = int(df_diag['review_comment_message'].isna().sum())
short_message = int((df_diag['review_comment_message'].str.len().fillna(0).between(1, 3)).sum())

# 6) Casos potencialmente ambiguos en duplicados por review_id
review_groups = df_diag.groupby('review_id', dropna=False).agg(
    n_orders=('order_id', 'nunique'),
    n_scores=('review_score', 'nunique')
)
multi_order_review = int((review_groups['n_orders'] > 1).sum())
multi_score_review = int((review_groups['n_scores'] > 1).sum())

# 7) Matriz de decisiones sugeridas (accion por regla)
rules = pd.DataFrame([
    {'criterio': 'Fila sin review_id', 'afectadas': missing_review_id, 'accion_sugerida': 'eliminar'},
    {'criterio': 'Fila sin order_id', 'afectadas': missing_order_id, 'accion_sugerida': 'eliminar'},
    {'criterio': 'Duplicado exacto', 'afectadas': dup_exact, 'accion_sugerida': 'eliminar'},
    {'criterio': 'Duplicado por review_id', 'afectadas': dup_review_id, 'accion_sugerida': 'marcar bandera (no eliminar directo)'},
    {'criterio': 'Duplicado por (review_id, order_id)', 'afectadas': dup_pair, 'accion_sugerida': 'revisar y posible deduplicacion puntual'},
    {'criterio': 'review_score no numerico', 'afectadas': invalid_score_type, 'accion_sugerida': 'imputar conservador'},
    {'criterio': 'review_score fuera de [1,5]', 'afectadas': score_out_of_range, 'accion_sugerida': 'clip al rango'},
    {'criterio': 'review_score no entero', 'afectadas': score_non_integer, 'accion_sugerida': 'round'},
    {'criterio': 'review_answer < review_creation', 'afectadas': answer_before_creation, 'accion_sugerida': 'marcar bandera temporal'},
    {'criterio': 'review_comment_title nulo', 'afectadas': missing_title, 'accion_sugerida': 'conservar (campo opcional)'},
    {'criterio': 'review_comment_message nulo', 'afectadas': missing_message, 'accion_sugerida': 'conservar (campo opcional)'},
    {'criterio': 'Placeholder en title', 'afectadas': placeholder_title, 'accion_sugerida': 'convertir a NA'},
    {'criterio': 'Placeholder en message', 'afectadas': placeholder_message, 'accion_sugerida': 'convertir a NA'},
    {'criterio': 'Mensaje muy corto (1-3 chars)', 'afectadas': short_message, 'accion_sugerida': 'mantener y marcar para analisis'}
])
rules['pct'] = (rules['afectadas'] / n * 100).round(2)

print(f'Filas totales: {n:,}')
print('Duplicados exactos:', dup_exact)
print('Duplicados por review_id:', dup_review_id)
print('Duplicados por (review_id, order_id):', dup_pair)
print('Review_id en multiples ordenes:', multi_order_review)
print('Review_id con multiples scores:', multi_score_review)

display(missing)
display(df_diag['review_score'].value_counts(dropna=False).sort_index().to_frame('count'))
display(rules.sort_values(by=['afectadas', 'criterio'], ascending=[False, True]).reset_index(drop=True))
display(df_diag.describe(include='all').T)

Filas totales: 99,224
Duplicados exactos: 0
Duplicados por review_id: 814
Duplicados por (review_id, order_id): 0
Review_id en multiples ordenes: 789
Review_id con multiples scores: 0


,missing_count,missing_pct,criticidad
review_id,0,0.00,alta
order_id,0,0.00,alta
review_score,0,0.00,alta
review_comment_title,87656,88.34,baja (campo opcional)
review_comment_message,58247,58.70,baja (campo opcional)
review_creation_date,0,0.00,alta
review_answer_timestamp,0,0.00,alta


,count
review_score,
1,11424
2,3151
3,8179
4,19142
5,57328


,criterio,afectadas,accion_sugerida,pct
0,review_comment_title nulo,87656,conservar (campo opcional),88.34
1,review_comment_message nulo,58247,conservar (campo opcional),58.70
2,Duplicado por review_id,814,marcar bandera (no eliminar directo),0.82
3,Mensaje muy corto (1-3 chars),809,mantener y marcar para analisis,0.82
4,Placeholder en message,4,convertir a NA,0.00
5,Placeholder en title,1,convertir a NA,0.00
6,Duplicado exacto,0,eliminar,0.00
7,"Duplicado por (review_id, order_id)",0,revisar y posible deduplicacion puntual,0.00
8,Fila sin order_id,0,eliminar,0.00
9,Fila sin review_id,0,eliminar,0.00


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
review_id,99224,98410,c444278834184f72b1484dfe47de7f97,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_id,99224,98673,c88b1d1b157a9999ce368f218a407141,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
review_score,99224.0,NaN,NaN,NaN,4.086421,1.347579,1.0,4.0,5.0,5.0,5.0
review_comment_title,11568,4179,Recomendo,485,NaN,NaN,NaN,NaN,NaN,NaN,NaN
review_comment_message,40977,35617,Muito bom,305,NaN,NaN,NaN,NaN,NaN,NaN,NaN
review_creation_date,99224,636,2017-12-19 00:00:00,463,NaN,NaN,NaN,NaN,NaN,NaN,NaN
review_answer_timestamp,99224,98248,2017-06-15 23:21:05,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Limpieza conservadora
df_clean = df.copy()

# 1) Estandarizar nombres de columnas
df_clean.columns = [c.strip().lower() for c in df_clean.columns]

# 2) Limpiar campos de texto de identificadores
id_cols = ['review_id', 'order_id']
for col in id_cols:
    df_clean[col] = df_clean[col].astype('string').str.strip()

# 3) Limpiar texto libre sin borrar informacion
text_cols = ['review_comment_title', 'review_comment_message']
for col in text_cols:
    df_clean[col] = df_clean[col].astype('string').str.replace(r'\s+', ' ', regex=True).str.strip()
    df_clean[col] = df_clean[col].replace({'': pd.NA, 'nan': pd.NA})

# 3b) Convertir placeholders a NA (detectados en diagnostico)
placeholder_pattern = r'^(?:no\s*comment|n/?a|\.+|-+|\?+|test|xxx)$'
for col in text_cols:
    mask_placeholder = df_clean[col].str.match(placeholder_pattern, case=False, na=False)
    print(f'Placeholders detectados en {col}: {int(mask_placeholder.sum())}')
    df_clean.loc[mask_placeholder, col] = pd.NA

missing_title_after_clean = int(df_clean['review_comment_title'].isna().sum())
missing_message_after_clean = int(df_clean['review_comment_message'].isna().sum())

# 3c) Bandera para mensajes muy cortos (1-3 chars) — mantener y marcar
df_clean['flag_short_message'] = (
    df_clean['review_comment_message'].notna()
    & (df_clean['review_comment_message'].str.len() <= 3)
)
print(f'Mensajes muy cortos (1-3 chars): {int(df_clean["flag_short_message"].sum())}')

# 4) Convertir review_score a numerico e imputar de forma conservadora
df_clean['review_score'] = pd.to_numeric(df_clean['review_score'], errors='coerce')
na_score = int(df_clean['review_score'].isna().sum())
if na_score > 0:
    median_score = int(round(df_clean['review_score'].median(skipna=True)))
    median_score = min(5, max(1, median_score))
else:
    median_score = 5
df_clean['review_score'] = df_clean['review_score'].fillna(median_score)

# 5) Ajustar review_score al rango valido [1, 5]
out_of_range_before = int(((df_clean['review_score'] < 1) | (df_clean['review_score'] > 5)).sum())
df_clean['review_score'] = df_clean['review_score'].clip(1, 5).round().astype('int64')

# 6) Eliminar solo registros sin llaves minimas
missing_review_id = df_clean['review_id'].isna() | df_clean['review_id'].isin(['', 'nan'])
missing_order_id = df_clean['order_id'].isna() | df_clean['order_id'].isin(['', 'nan'])
rows_without_keys = int((missing_review_id | missing_order_id).sum())
df_clean = df_clean[~(missing_review_id | missing_order_id)].copy()

# 7) Fechas: parsear y conservar nulos si existen
df_clean['review_creation_date'] = pd.to_datetime(df_clean['review_creation_date'], errors='coerce')
df_clean['review_answer_timestamp'] = pd.to_datetime(df_clean['review_answer_timestamp'], errors='coerce')
invalid_creation_date = int(df_clean['review_creation_date'].isna().sum())
invalid_answer_date = int(df_clean['review_answer_timestamp'].isna().sum())

# 8) Bandera de calidad temporal (sin eliminar)
df_clean['flag_answer_before_creation'] = (
    df_clean['review_answer_timestamp'].notna()
    & df_clean['review_creation_date'].notna()
    & (df_clean['review_answer_timestamp'] < df_clean['review_creation_date'])
)

# 9) Banderas para review_id repetido (no se elimina)
df_clean['flag_review_id_duplicated'] = df_clean.duplicated(subset=['review_id'], keep=False)
df_clean['flag_review_id_multi_order'] = df_clean.groupby('review_id')['order_id'].transform('nunique').gt(1)

# 10) Eliminar duplicados exactos al final (sin contar columnas de banderas)
flag_cols = [c for c in df_clean.columns if c.startswith('flag_')]
original_cols = [c for c in df_clean.columns if c not in flag_cols]
before_dup = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=original_cols)
removed_dup = before_dup - len(df_clean)

# --- Reporte final ---
print(f'\n{"="*50}')
print(f'REPORTE DE LIMPIEZA')
print(f'{"="*50}')
print(f'Filas sin llaves minimas eliminadas: {rows_without_keys}')
print(f'Duplicados exactos eliminados: {removed_dup}')
print(f'Review_id repetidos (conservados): {int(df_clean["flag_review_id_duplicated"].sum())}')
print(f'Review_id en multiples ordenes: {int(df_clean["flag_review_id_multi_order"].sum())}')
print(f'Nulos imputados en review_score: {na_score}')
print(f'Review_score fuera de rango (clip): {out_of_range_before}')
print(f'Comentarios sin titulo (esperado): {missing_title_after_clean}')
print(f'Comentarios sin mensaje (esperado): {missing_message_after_clean}')
print(f'Mensajes cortos (flag): {int(df_clean["flag_short_message"].sum())}')
print(f'Fechas creacion no parseables: {invalid_creation_date}')
print(f'Fechas respuesta no parseables: {invalid_answer_date}')
print(f'Filas con answer < creation: {int(df_clean["flag_answer_before_creation"].sum())}')
print(f'Filas finales: {len(df_clean):,}')

Placeholders detectados en review_comment_title: 21
Placeholders detectados en review_comment_message: 99
Mensajes muy cortos (1-3 chars): 724

REPORTE DE LIMPIEZA
Filas sin llaves minimas eliminadas: 0
Duplicados exactos eliminados: 0
Review_id repetidos (conservados): 1603
Review_id en multiples ordenes: 1603
Nulos imputados en review_score: 0
Review_score fuera de rango (clip): 0
Comentarios sin titulo (esperado): 87679
Comentarios sin mensaje (esperado): 58373
Mensajes cortos (flag): 724
Fechas creacion no parseables: 0
Fechas respuesta no parseables: 0
Filas con answer < creation: 0
Filas finales: 99,224


In [ ]:
# Revision final de calidad
display(df_clean.head())
display(df_clean.isna().sum().to_frame('missing_count'))
display(df_clean.describe(include='all').T)

print('Duplicados exactos finales:', int(df_clean.duplicated().sum()))
print('Duplicados por review_id finales:', int(df_clean.duplicated(subset=['review_id']).sum()))
print('Filas marcadas con review_id repetido:', int(df_clean['flag_review_id_duplicated'].sum()))
print('Filas marcadas con review_id en multiples ordenes:', int(df_clean['flag_review_id_multi_order'].sum()))

# Revisar qué convirtió a NA el patrón de placeholders
mask = df_clean['review_comment_message'].isna() & df['review_comment_message'].notna()
print(df.loc[mask, 'review_comment_message'].value_counts().head(20))

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,flag_short_message,flag_answer_before_creation,flag_review_id_duplicated,flag_review_id_multi_order
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,<NA>,<NA>,2018-01-18,2018-01-18 21:46:59,False,False,False,False
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,<NA>,<NA>,2018-03-10,2018-03-11 03:05:13,False,False,False,False
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,<NA>,<NA>,2018-02-17,2018-02-18 14:36:24,False,False,False,False
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,<NA>,Recebi bem antes do prazo estipulado.,2017-04-21,2017-04-21 22:02:06,False,False,False,False
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,<NA>,Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa,2018-03-01,2018-03-02 10:26:53,False,False,False,False


,missing_count
review_id,0
order_id,0
review_score,0
review_comment_title,87679
review_comment_message,58373
review_creation_date,0
review_answer_timestamp,0
flag_short_message,0
flag_answer_before_creation,0
flag_review_id_duplicated,0


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
review_id,99224,98410,c444278834184f72b1484dfe47de7f97,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_id,99224,98673,c88b1d1b157a9999ce368f218a407141,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
review_score,99224.0,NaN,NaN,NaN,4.086421,1.0,4.0,5.0,5.0,5.0,1.347579
review_comment_title,11545,4166,Recomendo,485,NaN,NaN,NaN,NaN,NaN,NaN,NaN
review_comment_message,40851,35597,Muito bom,305,NaN,NaN,NaN,NaN,NaN,NaN,NaN
review_creation_date,99224,NaN,NaN,NaN,2018-01-12 20:49:23.948238,2016-10-02 00:00:00,2017-09-23 00:00:00,2018-02-02 00:00:00,2018-05-16 00:00:00,2018-08-31 00:00:00,NaN
review_answer_timestamp,99224,NaN,NaN,NaN,2018-01-16 00:23:56.977939,2016-10-07 18:32:28,2017-09-27 01:53:27.250000,2018-02-04 22:41:47.500000,2018-05-20 12:11:21.500000,2018-10-29 12:27:35,NaN
flag_short_message,99224,2,False,98500,NaN,NaN,NaN,NaN,NaN,NaN,NaN
flag_answer_before_creation,99224,1,False,99224,NaN,NaN,NaN,NaN,NaN,NaN,NaN
flag_review_id_duplicated,99224,2,False,97621,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Duplicados exactos finales: 0
Duplicados por review_id finales: 814
Filas marcadas con review_id repetido: 1603
Filas marcadas con review_id en multiples ordenes: 1603
review_comment_message
.                                           51
\r\n                                        14
...                                         11
                                             9
..                                           6
xxx                                          4
\r\n\r\n                                     3
.....                                        3
?                                            3
-                                            3
....                                         3
.........                                    2
..                                           2
Xxx                                          2
N/a                                          1
\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n     1
\r\n.                                        1
????      

In [ ]:
# Exportar dataset limpio
output_path = 'olist_order_reviews_dataset_clean.csv'
df_clean.to_csv(output_path, index=False)
print(f'Archivo guardado en: {output_path}')

Archivo guardado en: olist_order_reviews_dataset_clean.csv


---
# 📂 Notebook: `limpieza_sellers_data.ipynb`
---

# Limpieza de `olist_seller_dataset.csv`

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
# Carga de datos
# dtype str en seller_zip_code_prefix para preservar ceros iniciales
# Se prueban varias rutas para compatibilidad local, Docker y distintos CWD de Jupyter
import os

_FILENAME = 'olist_sellers_dataset.csv'
_CANDIDATES = [
    os.path.join('..', 'data', _FILENAME),   # CWD = notebooks/
    os.path.join('data', _FILENAME),          # CWD = raíz del proyecto
    os.path.join('work', 'data', _FILENAME),  # CWD = /home/jovyan (Docker)
]

_data_path = next((p for p in _CANDIDATES if os.path.exists(p)), None)
if _data_path is None:
    raise FileNotFoundError(
        f"No se encontró '{_FILENAME}' en ninguna ruta candidata: {_CANDIDATES}"
    )

print(f"Cargando datos desde: {os.path.abspath(_data_path)}")
df = pd.read_csv(_data_path, dtype={'seller_zip_code_prefix': str})
df.head()



Cargando datos desde: /home/jovyan/work/data/olist_sellers_dataset.csv


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,04195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


## 1. Exploración inicial

In [ ]:
print("Dimensiones:", df.shape)
print("\nTipos de datos:")
print(df.dtypes)
print("\nPrimeras filas:")
df.head()


Dimensiones: (3095, 4)

Tipos de datos:
seller_id                 object
seller_zip_code_prefix    object
seller_city               object
seller_state              object
dtype: object

Primeras filas:


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,04195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [ ]:
print("=== Valores nulos por columna ===")
print(df.isnull().sum())
print("\nFilas duplicadas:", df.duplicated().sum())
print("IDs de vendedor duplicados:", df['seller_id'].duplicated().sum())
print("\nEstados únicos:", sorted(df['seller_state'].unique().tolist()))
print("\nCiudades únicas:", df['seller_city'].nunique())


=== Valores nulos por columna ===
seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

Filas duplicadas: 0
IDs de vendedor duplicados: 0

Estados únicos: ['AC', 'AM', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA', 'MG', 'MS', 'MT', 'PA', 'PB', 'PE', 'PI', 'PR', 'RJ', 'RN', 'RO', 'RS', 'SC', 'SE', 'SP']

Ciudades únicas: 611


## 2. Análisis de `seller_city`

La columna `seller_city` presenta varios tipos de inconsistencias que se detallan a continuación.

In [ ]:
import re
import unicodedata

cities = df['seller_city']

problemas = {
    'Con barra (ciudad/estado o ciudad/ciudad)': cities[cities.str.contains(r'[/\\]', regex=True, na=False)].unique().tolist(),
    'Con coma (ciudad, estado, país)':           cities[cities.str.contains(r',', na=False)].unique().tolist(),
    'Con paréntesis':                            cities[cities.str.contains(r'\(', na=False)].unique().tolist(),
    'Con acentos':                               cities[cities.str.contains(r'[àáâãäéêëíïóôõöúüçñ]', regex=True, na=False)].unique().tolist(),
    'Con apóstrofe especial (´)':               cities[cities.str.contains('´', na=False)].unique().tolist(),
    'Solo dígitos':                              cities[cities.str.fullmatch(r'\d+', na=False)].unique().tolist(),
    'Email':                                     cities[cities.str.contains('@', na=False)].unique().tolist(),
    'Abreviatura de estado (sp, rj…)':          cities[cities.isin(['sp', 'rj', 'es', 'pr', 'mg'])].unique().tolist(),
    'Estado incluido en texto (ej. brasilia df)': cities[cities.str.contains(r'\bdf\b|\bmg\b|\brs\b', regex=True, na=False)
                                                         & ~cities.str.contains(r'[/\\,]', na=False)].unique().tolist(),
}

for tipo, vals in problemas.items():
    if vals:
        print(f"\n[{tipo}] ({len(vals)} valores únicos):")
        for v in vals:
            print(f"    {v!r}")



[Con barra (ciudad/estado o ciudad/ciudad)] (16 valores únicos):
    'auriflama/sp'
    'sao paulo / sao paulo'
    'cariacica / es'
    'sbc/sp'
    'santo andre/sao paulo'
    'sp / sp'
    'maua/sao paulo'
    'mogi das cruzes / sp'
    'rio de janeiro \\rio de janeiro'
    'barbacena/ minas gerais'
    'rio de janeiro / rio de janeiro'
    'pinhais/pr'
    'ribeirao preto / sao paulo'
    'carapicuiba / sao paulo'
    'sao sebastiao da grama/sp'
    'jacarei / sao paulo'

[Con coma (ciudad, estado, país)] (2 valores únicos):
    'novo hamburgo, rio grande do sul, brasil'
    'rio de janeiro, rio de janeiro, brasil'

[Con paréntesis] (1 valores únicos):
    "arraial d'ajuda (porto seguro)"

[Con apóstrofe especial (´)] (1 valores únicos):
    'santa barbara d´oeste'

[Solo dígitos] (1 valores únicos):
    '04482255'

[Email] (1 valores únicos):
    'vendas@creditparts.com.br'

[Abreviatura de estado (sp, rj…)] (1 valores únicos):
    'sp'

[Estado incluido en texto (ej. brasilia df

## 3. Limpieza de `seller_city`

Pasos aplicados en orden:
1. Valores irrecuperables (solo dígitos, email) → `NaN`
2. Eliminar acentos para coherencia con el resto del dataset
3. Normalizar apóstrofes especiales (`´` → `'`)
4. Extraer parte antes de `/` o `\` (formatos `ciudad/estado`)
5. Extraer parte antes de la primera coma (formatos `ciudad, estado, país`)
6. Eliminar contenido entre paréntesis
7. Correcciones puntuales: abreviaturas (`sp` → `sao paulo`) y sufijos de estado (`brasilia df` → `brasilia`)

In [ ]:
REPLACEMENTS = {
    'sp':          'sao paulo',
    'brasilia df': 'brasilia',
}

def clean_city(city: str) -> str:
    if not isinstance(city, str):
        return np.nan

    city = city.strip().lower()

    # 1. Valores irrecuperables: solo dígitos o dirección de email
    if re.fullmatch(r'\d+', city) or '@' in city:
        return np.nan

    # 2. Eliminar acentos para coherencia con el resto del dataset
    city = unicodedata.normalize('NFKD', city)
    city = ''.join(c for c in city if not unicodedata.combining(c))

    # 3. Normalizar apóstrofes especiales
    city = city.replace('´', "'")

    # 4. Extraer parte antes de / o \ (formatos ciudad/estado o ciudad\estado)
    city = re.split(r'[/\\]', city)[0].strip()

    # 5. Extraer parte antes de la primera coma (formatos ciudad, estado, país)
    city = city.split(',')[0].strip()

    # 6. Eliminar contenido entre paréntesis
    city = re.sub(r'\s*\(.*?\)', '', city).strip()

    # 7. Correcciones puntuales
    city = REPLACEMENTS.get(city, city)

    return city


# Guardar valores originales para el reporte
df['seller_city_original'] = df['seller_city']

# Aplicar limpieza
df['seller_city'] = df['seller_city'].apply(clean_city)

print("Limpieza aplicada. Resumen de cambios:")
mascara_cambio = df['seller_city'] != df['seller_city_original']
print(f"  Filas modificadas : {mascara_cambio.sum()}")
print(f"  Nuevos NaN        : {df['seller_city'].isnull().sum()}")


Limpieza aplicada. Resumen de cambios:
  Filas modificadas : 29
  Nuevos NaN        : 2


In [ ]:
# Mostrar las transformaciones realizadas
cambios = df[mascara_cambio][['seller_city_original', 'seller_city', 'seller_state']].drop_duplicates()
cambios.columns = ['ciudad_original', 'ciudad_limpia', 'estado']
cambios.sort_values('ciudad_original').reset_index(drop=True)


,ciudad_original,ciudad_limpia,estado
0,04482255,NaN,RJ
1,arraial d'ajuda (porto seguro),arraial d'ajuda,BA
2,auriflama/sp,auriflama,SP
3,barbacena/ minas gerais,barbacena,MG
4,brasilia df,brasilia,DF
5,carapicuiba / sao paulo,carapicuiba,SP
6,cariacica / es,cariacica,ES
7,jacarei / sao paulo,jacarei,SP
8,maua/sao paulo,maua,SP
9,mogi das cruzes / sp,mogi das cruzes,SP


## 4. Verificación de `seller_zip_code_prefix`

El código postal fue cargado como `str` para preservar ceros iniciales. Se verifica que todos los valores tengan exactamente 5 dígitos y se aplica zero-padding donde sea necesario.

In [ ]:
zips = df['seller_zip_code_prefix']

no_numericos = zips[~zips.str.fullmatch(r'\d+')].unique().tolist()
longitud_incorrecta = zips[zips.str.len() != 5].unique().tolist()

print(f"Zip codes no numéricos      : {len(no_numericos)}")
print(f"Zip codes con longitud ≠ 5  : {len(longitud_incorrecta)}")
print(f"Con cero inicial            : {(zips.str[0] == '0').sum()}")

# Aplicar zero-padding por si algún valor tuviese menos de 5 dígitos
df['seller_zip_code_prefix'] = zips.str.zfill(5)

print("\nEjemplos de zip_code_prefix tras normalización:")
print(df['seller_zip_code_prefix'].head(10).tolist())


Zip codes no numéricos      : 0
Zip codes con longitud ≠ 5  : 0
Con cero inicial            : 1027

Ejemplos de zip_code_prefix tras normalización:
['13023', '13844', '20031', '04195', '12914', '20920', '55325', '16304', '01529', '80310']


## 5. Reporte final

In [ ]:
ciudades_antes = df['seller_city_original'].nunique()
df_clean = df.drop(columns=['seller_city_original'])

print("=" * 50)
print("REPORTE DE LIMPIEZA")
print("=" * 50)
print(f"\nFilas totales            : {len(df_clean)}")
print(f"Columnas                 : {list(df_clean.columns)}")
print(f"\nValores nulos tras limpieza:")
print('Nulos por columna en df_clean:')
print(df_clean.isnull().sum().to_string())
print(f"\nFilas con seller_city nulo (irrecuperables): {df_clean['seller_city'].isnull().sum()}")
print(f"Ciudades únicas (antes)  : {ciudades_antes}")
print(f"Ciudades únicas (después): {df_clean['seller_city'].nunique()}")
print(f"\nEstados únicos           : {sorted(df_clean['seller_state'].unique().tolist())}")
print(f"\nTop 10 ciudades (limpias):")
print(df_clean['seller_city'].value_counts().head(10).to_string())


REPORTE DE LIMPIEZA

Filas totales            : 3095
Columnas                 : ['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']

Valores nulos tras limpieza:
seller_id                 0
seller_zip_code_prefix    0
seller_city               2
seller_state              0

Filas con seller_city nulo (irrecuperables): 2
Ciudades únicas (antes)  : 611
Ciudades únicas (después): 588

Estados únicos           : ['AC', 'AM', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA', 'MG', 'MS', 'MT', 'PA', 'PB', 'PE', 'PI', 'PR', 'RJ', 'RN', 'RO', 'RS', 'SC', 'SE', 'SP']

Top 10 ciudades (limpias):
seller_city
sao paulo         701
curitiba          127
rio de janeiro     99
belo horizonte     68
ribeirao preto     53
guarulhos          50
ibitinga           49
santo andre        46
campinas           41
maringa            40


## 6. Exportar dataset limpio

In [ ]:
# Guardar en la misma carpeta que el archivo original
output_path = os.path.join(os.path.dirname(_data_path), 'olist_sellers_dataset_clean.csv')
df_clean.to_csv(output_path, index=False)
print(f"Dataset limpio guardado en: {os.path.abspath(output_path)}")
print(f"Forma final: {df_clean.shape}")


Dataset limpio guardado en: /home/jovyan/work/data/olist_sellers_dataset_clean.csv
Forma final: (3095, 4)
